fazer a tratagem das tabelas Olist 

In [0]:
df = spark.table("olist.bronze.orders")
display(df.limit(3))
df.printSchema()

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: string (nullable = true)
 |-- order_approved_at: string (nullable = true)
 |-- order_delivered_carrier_date: string (nullable = true)
 |-- order_delivered_customer_date: string (nullable = true)
 |-- order_estimated_delivery_date: string (nullable = true)



In [0]:
columns_data = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for i in columns_data:
    df = df.withColumn(i, df[i].cast("timestamp"))

df = df.dropDuplicates(["order_id"])

df.write.mode("overwrite").saveAsTable("olist.silver.orders")

total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.orders ===")
print(f"Total de linhas: {total}")

print("\n1) Pedidos duplicados (order_id repetido). Esperado: 0")
duplicados = total - df.select("order_id").distinct().count()
print(f"   Duplicados: {duplicados}")

print("\n2) Nulos por coluna")
for c in df.columns:
    nulos = df.filter(df[c].isNull()).count()
    print(f"   {c}: {nulos} nulos")

print("\n3) Quantidade de pedidos por status")
display(df.groupBy("order_status").count().orderBy("count", ascending=False).limit(3))

print("\n4) Pedidos SEM data de entrega, por status")
print("   Normal para cancelados e em andamento. Estranho se for 'delivered'.")
display(
    df.filter(df.order_delivered_customer_date.isNull())
      .groupBy("order_status")
      .count()
      .orderBy("count", ascending=False)
      .limit(3)
)

print("\n5) Datas impossíveis. Esperado: 0 nas duas")
antes_compra = df.filter(df.order_delivered_customer_date < df.order_purchase_timestamp).count()
print(f"   Entregues antes da compra: {antes_compra}")
aprovado_antes = df.filter(df.order_approved_at < df.order_purchase_timestamp).count()
print(f"   Aprovados antes da compra: {aprovado_antes}")

=== CHECAGEM DE QUALIDADE: olist.silver.orders ===
Total de linhas: 99441

1) Pedidos duplicados (order_id repetido). Esperado: 0
   Duplicados: 0

2) Nulos por coluna
   order_id: 0 nulos
   customer_id: 0 nulos
   order_status: 0 nulos
   order_purchase_timestamp: 0 nulos
   order_approved_at: 160 nulos
   order_delivered_carrier_date: 1783 nulos
   order_delivered_customer_date: 2965 nulos
   order_estimated_delivery_date: 0 nulos

3) Quantidade de pedidos por status


order_status,count
delivered,96478
shipped,1107
canceled,625



4) Pedidos SEM data de entrega, por status
   Normal para cancelados e em andamento. Estranho se for 'delivered'.


order_status,count
shipped,1107
canceled,619
unavailable,609



5) Datas impossíveis. Esperado: 0 nas duas
   Entregues antes da compra: 0
   Aprovados antes da compra: 0


**Observação sobre `silver.orders`:** 8 pedidos com status `delivered` não têm data de entrega ao cliente (`order_delivered_customer_date`). Esses registros foram mantidos na silver, que guarda dados limpos e de uso geral, e serão filtrados apenas nas golds em que a data é necessária, como no modelo de atraso. Os demais pedidos sem data de entrega (2.957) são esperados, pois estão em andamento, cancelados ou indisponíveis.

In [0]:
df_items = spark.table("olist.bronze.order_items")
display(df_items.limit(3))
df_items.printSchema()

order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: string (nullable = true)
 |-- price: string (nullable = true)
 |-- freight_value: string (nullable = true)



In [0]:
colunas_int = ["order_item_id"]
colunas_double = ["price", "freight_value"]

for c in colunas_int:
    df_items = df_items.withColumn(c, df_items[c].cast("int"))

for c in colunas_double:
    df_items = df_items.withColumn(c, df_items[c].cast("double"))

df_items = df_items.withColumn("shipping_limit_date", df_items["shipping_limit_date"].cast("timestamp"))

df_items = df_items.dropDuplicates(["order_id", "order_item_id"])

df_items.write.mode("overwrite").saveAsTable("olist.silver.order_items")

In [0]:
spark.table("olist.silver.order_items").printSchema()
print(spark.table("olist.silver.order_items").count())


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

112650


In [0]:
df = spark.table("olist.silver.order_items")
total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.order_items ===")
print(f"Total de linhas: {total}")

print("\n1) Itens duplicados (order_id + order_item_id). Esperado: 0")
distintos = df.select("order_id", "order_item_id").distinct().count()
print(f"   Duplicados: {total - distintos}")

print("\n2) Nulos por coluna. Esperado: 0 em todas")
print("   Se aparecer nulo em coluna convertida, o cast falhou para essas linhas.")
for c in df.columns:
    print(f"   {c}: {df.filter(df[c].isNull()).count()} nulos")

print("\n3) Valores inválidos de preço e frete")
print(f"   Preço <= 0 (esperado 0): {df.filter(df.price <= 0).count()}")
print(f"   Frete < 0 (esperado 0): {df.filter(df.freight_value < 0).count()}")
print(f"   Frete = 0 (pode ser frete grátis, normal): {df.filter(df.freight_value == 0).count()}")

print("\n4) Itens cujo pedido não existe em silver.orders. Esperado: 0")
orders = spark.table("olist.silver.orders")
print(f"   Itens órfãos: {df.join(orders, 'order_id', 'left_anti').count()}")

print("\n5) Pedidos de silver.orders sem nenhum item")
print("   Pode acontecer em pedidos cancelados ou indisponíveis.")
print(f"   Pedidos sem itens: {orders.join(df, 'order_id', 'left_anti').count()}")

print("\n6) Resumo de preço e frete (mínimo, média, máximo)")
display(df.select("price", "freight_value").summary("min", "mean", "max").limit(3))

=== CHECAGEM DE QUALIDADE: olist.silver.order_items ===
Total de linhas: 112650

1) Itens duplicados (order_id + order_item_id). Esperado: 0
   Duplicados: 0

2) Nulos por coluna. Esperado: 0 em todas
   Se aparecer nulo em coluna convertida, o cast falhou para essas linhas.
   order_id: 0 nulos
   order_item_id: 0 nulos
   product_id: 0 nulos
   seller_id: 0 nulos
   shipping_limit_date: 0 nulos
   price: 0 nulos
   freight_value: 0 nulos

3) Valores inválidos de preço e frete
   Preço <= 0 (esperado 0): 0
   Frete < 0 (esperado 0): 0
   Frete = 0 (pode ser frete grátis, normal): 383

4) Itens cujo pedido não existe em silver.orders. Esperado: 0
   Itens órfãos: 0

5) Pedidos de silver.orders sem nenhum item
   Pode acontecer em pedidos cancelados ou indisponíveis.
   Pedidos sem itens: 775

6) Resumo de preço e frete (mínimo, média, máximo)


summary,price,freight_value
min,0.85,0.0
mean,120.653739014649,19.99031992898357
max,6735.0,409.68


In [0]:
df_pay = spark.table("olist.bronze.order_payments")
display(df_pay.limit(3))
df_pay.printSchema()

order_id,payment_sequential,payment_type,payment_installments,payment_value
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71


root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: string (nullable = true)
 |-- payment_value: string (nullable = true)



In [0]:
colunas_int = ["payment_sequential", "payment_installments"]

for c in colunas_int:
    df_pay = df_pay.withColumn(c, df_pay[c].cast("int"))

df_pay = df_pay.withColumn("payment_value", df_pay["payment_value"].cast("double"))

df_pay = df_pay.dropDuplicates(["order_id", "payment_sequential"])

df_pay.write.mode("overwrite").saveAsTable("olist.silver.order_payments")

In [0]:
df = spark.table("olist.silver.order_payments")
total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.order_payments ===")
print(f"Total de linhas: {total}  (esperado: 103886)")

print("\n1) Pagamentos duplicados (order_id + payment_sequential). Esperado: 0")
distintos = df.select("order_id", "payment_sequential").distinct().count()
print(f"   Duplicados: {total - distintos}")

print("\n2) Nulos por coluna. Esperado: 0 em todas")
print("   Se aparecer nulo em coluna convertida, o cast falhou para essas linhas.")
for c in df.columns:
    print(f"   {c}: {df.filter(df[c].isNull()).count()} nulos")

print("\n3) Tipos de pagamento existentes")
print("   Confira se há algum valor estranho, como 'not_defined'.")
display(df.groupBy("payment_type").count().orderBy("count", ascending=False).limit(3))

print("\n4) Valores e parcelas suspeitos")
print(f"   Valor < 0 (esperado 0): {df.filter(df.payment_value < 0).count()}")
print(f"   Valor = 0 (investigar): {df.filter(df.payment_value == 0).count()}")
print(f"   Parcelas = 0 (investigar): {df.filter(df.payment_installments == 0).count()}")

print("\n5) Pagamentos de pedidos que não existem em silver.orders. Esperado: 0")
orders = spark.table("olist.silver.orders")
print(f"   Pagamentos órfãos: {df.join(orders, 'order_id', 'left_anti').count()}")

print("\n6) Pedidos de silver.orders sem nenhum pagamento")
print(f"   Pedidos sem pagamento: {orders.join(df, 'order_id', 'left_anti').count()}")

print("\n7) Resumo do valor pago (mínimo, média, máximo)")
display(df.select("payment_value", "payment_installments").summary("min", "mean", "max").limit(3))

=== CHECAGEM DE QUALIDADE: olist.silver.order_payments ===
Total de linhas: 103886  (esperado: 103886)

1) Pagamentos duplicados (order_id + payment_sequential). Esperado: 0
   Duplicados: 0

2) Nulos por coluna. Esperado: 0 em todas
   Se aparecer nulo em coluna convertida, o cast falhou para essas linhas.
   order_id: 0 nulos
   payment_sequential: 0 nulos
   payment_type: 0 nulos
   payment_installments: 0 nulos
   payment_value: 0 nulos

3) Tipos de pagamento existentes
   Confira se há algum valor estranho, como 'not_defined'.


payment_type,count
credit_card,76795
boleto,19784
voucher,5775



4) Valores e parcelas suspeitos
   Valor < 0 (esperado 0): 0
   Valor = 0 (investigar): 9
   Parcelas = 0 (investigar): 2

5) Pagamentos de pedidos que não existem em silver.orders. Esperado: 0
   Pagamentos órfãos: 0

6) Pedidos de silver.orders sem nenhum pagamento
   Pedidos sem pagamento: 1

7) Resumo do valor pago (mínimo, média, máximo)


summary,payment_value,payment_installments
min,0.0,0
mean,154.10038041699568,2.853348863176944
max,13664.08,24


Observação sobre silver.order_payments: 9 pagamentos têm valor zero, 2 têm parcelas igual a zero e 1 pedido de silver.orders não tem pagamento registrado. Os registros foram mantidos, e cada projeto decide na gold se precisa filtrá-los

In [0]:
df_rev = spark.table("olist.bronze.order_reviews")
display(df_rev.limit(3))
df_rev.printSchema()

review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24


root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)



In [0]:
df_rev = df_rev.withColumn("review_score", df_rev["review_score"].cast("int"))

colunas_data = ["review_creation_date", "review_answer_timestamp"]

for c in colunas_data:
    df_rev = df_rev.withColumn(c, df_rev[c].cast("timestamp"))

df_rev.write.mode("overwrite").saveAsTable("olist.silver.order_reviews")

In [0]:
df = spark.table("olist.silver.order_reviews")
total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.order_reviews ===")
print(f"Total de linhas: {total}  (esperado: 99224)")

print("\n1) Duplicidade")
ids_distintos = df.select("review_id").distinct().count()
pares_distintos = df.select("review_id", "order_id").distinct().count()
print(f"   review_id diferentes: {ids_distintos}")
print(f"   Linhas com review_id repetido: {total - ids_distintos}")
print(f"   Linhas repetidas considerando review_id + order_id (esperado: 0): {total - pares_distintos}")

print("\n2) Nulos por coluna")
print("   Nulo em título e mensagem é normal: o cliente pode dar só a nota.")
print("   Nas demais colunas o esperado é 0.")
for c in df.columns:
    print(f"   {c}: {df.filter(df[c].isNull()).count()} nulos")

print("\n3) Notas (review_score)")
print("   Esperado: só valores de 1 a 5")
display(df.groupBy("review_score").count().orderBy("review_score").limit(3))

print("\n4) Datas impossíveis. Esperado: 0")
resposta_antes = df.filter(df.review_answer_timestamp < df.review_creation_date).count()
print(f"   Respondidas antes de a avaliação ser criada: {resposta_antes}")

print("\n5) Avaliações de pedidos que não existem em silver.orders. Esperado: 0")
orders = spark.table("olist.silver.orders")
print(f"   Avaliações órfãs: {df.join(orders, 'order_id', 'left_anti').count()}")

print("\n6) Pedidos de silver.orders sem nenhuma avaliação")
print("   Normal: nem todo cliente avalia.")
print(f"   Pedidos sem avaliação: {orders.join(df, 'order_id', 'left_anti').count()}")

print("\n7) Pedidos com mais de uma avaliação")
por_pedido = df.groupBy("order_id").count().withColumnRenamed("count", "qtd_avaliacoes")
display(
    por_pedido.filter("qtd_avaliacoes > 1")
              .groupBy("qtd_avaliacoes").count()
              .withColumnRenamed("count", "pedidos")
              .orderBy("qtd_avaliacoes")
              .limit(3)
)

=== CHECAGEM DE QUALIDADE: olist.silver.order_reviews ===
Total de linhas: 99224  (esperado: 99224)

1) Duplicidade
   review_id diferentes: 98410
   Linhas com review_id repetido: 814
   Linhas repetidas considerando review_id + order_id (esperado: 0): 0

2) Nulos por coluna
   Nulo em título e mensagem é normal: o cliente pode dar só a nota.
   Nas demais colunas o esperado é 0.
   review_id: 0 nulos
   order_id: 0 nulos
   review_score: 0 nulos
   review_comment_title: 87656 nulos
   review_comment_message: 58247 nulos
   review_creation_date: 0 nulos
   review_answer_timestamp: 0 nulos

3) Notas (review_score)
   Esperado: só valores de 1 a 5


review_score,count
1,11424
2,3151
3,8179



4) Datas impossíveis. Esperado: 0
   Respondidas antes de a avaliação ser criada: 0

5) Avaliações de pedidos que não existem em silver.orders. Esperado: 0
   Avaliações órfãs: 0

6) Pedidos de silver.orders sem nenhuma avaliação
   Normal: nem todo cliente avalia.
   Pedidos sem avaliação: 768

7) Pedidos com mais de uma avaliação


qtd_avaliacoes,pedidos
2,543
3,4


In [0]:
df_prod = spark.table("olist.bronze.products")
display(df_prod.limit(3))
df_prod.printSchema()

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: string (nullable = true)
 |-- product_description_lenght: string (nullable = true)
 |-- product_photos_qty: string (nullable = true)
 |-- product_weight_g: string (nullable = true)
 |-- product_length_cm: string (nullable = true)
 |-- product_height_cm: string (nullable = true)
 |-- product_width_cm: string (nullable = true)



In [0]:
df_trad = spark.table("olist.bronze.category_translation")
display(df_trad.limit(3))
df_trad.printSchema()

product_category_name,product_category_name_english
beleza_saude,health_beauty
informatica_acessorios,computers_accessories
automotivo,auto


root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)



In [0]:
df_prod = (
    df_prod
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
)

colunas_int = [
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

for c in colunas_int:
    df_prod = df_prod.withColumn(c, df_prod[c].cast("int"))

df_trad = df_trad.dropDuplicates(["product_category_name"])

df_prod = df_prod.join(df_trad, "product_category_name", "left")

df_prod = df_prod.dropDuplicates(["product_id"])

df_prod.write.mode("overwrite").saveAsTable("olist.silver.products")

In [0]:
spark.table("olist.silver.products").printSchema()
print(spark.table("olist.silver.products").count())


root
 |-- product_category_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_name_length: integer (nullable = true)
 |-- product_description_length: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)
 |-- product_category_name_english: string (nullable = true)

32951


In [0]:
df = spark.table("olist.silver.products")
total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.products ===")
print(f"Total de linhas: {total}  (esperado: 32951)")

print("\n1) Produtos duplicados (product_id repetido). Esperado: 0")
print(f"   Duplicados: {total - df.select('product_id').distinct().count()}")

print("\n2) Nulos por coluna")
print("   Alguns produtos vêm com cadastro incompleto (sem categoria, nome, descrição e fotos).")
for c in df.columns:
    print(f"   {c}: {df.filter(df[c].isNull()).count()} nulos")

print("\n3) Categorias preenchidas mas SEM tradução para o inglês")
display(
    df.filter(df.product_category_name.isNotNull() & df.product_category_name_english.isNull())
      .groupBy("product_category_name").count()
      .orderBy("count", ascending=False)
      .limit(3)
)

print("\n4) Medidas inválidas (zero ou negativas). Esperado: 0")
for c in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    print(f"   {c} <= 0: {df.filter(df[c] <= 0).count()}")

print("\n5) Itens de silver.order_items cujo produto não existe em silver.products. Esperado: 0")
itens = spark.table("olist.silver.order_items")
print(f"   Itens sem produto: {itens.join(df, 'product_id', 'left_anti').count()}")

print("\n6) Produtos que nunca aparecem em nenhum item")
print(f"   Produtos sem venda: {df.join(itens, 'product_id', 'left_anti').count()}")

print("\n7) Resumo das medidas (mínimo, média, máximo)")
display(
    df.select("product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm")
      .summary("min", "mean", "max")
      .limit(3)
)


=== CHECAGEM DE QUALIDADE: olist.silver.products ===
Total de linhas: 32951  (esperado: 32951)

1) Produtos duplicados (product_id repetido). Esperado: 0
   Duplicados: 0

2) Nulos por coluna
   Alguns produtos vêm com cadastro incompleto (sem categoria, nome, descrição e fotos).
   product_category_name: 610 nulos
   product_id: 0 nulos
   product_name_length: 610 nulos
   product_description_length: 610 nulos
   product_photos_qty: 610 nulos
   product_weight_g: 2 nulos
   product_length_cm: 2 nulos
   product_height_cm: 2 nulos
   product_width_cm: 2 nulos
   product_category_name_english: 623 nulos

3) Categorias preenchidas mas SEM tradução para o inglês


product_category_name,count
portateis_cozinha_e_preparadores_de_alimentos,10
pc_gamer,3



4) Medidas inválidas (zero ou negativas). Esperado: 0
   product_weight_g <= 0: 4
   product_length_cm <= 0: 0
   product_height_cm <= 0: 0
   product_width_cm <= 0: 0

5) Itens de silver.order_items cujo produto não existe em silver.products. Esperado: 0
   Itens sem produto: 0

6) Produtos que nunca aparecem em nenhum item
   Produtos sem venda: 0

7) Resumo das medidas (mínimo, média, máximo)


summary,product_weight_g,product_length_cm,product_height_cm,product_width_cm
min,0,7,2,6
mean,2276.4724877841513,30.81507784758263,16.937661234028347,23.196728277034204
max,40425,105,105,118


In [0]:
df_geo = spark.table("olist.bronze.geolocation")
display(df_geo.limit(3))
df_geo.printSchema()

geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP


root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: string (nullable = true)
 |-- geolocation_lng: string (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



In [0]:
df_geo = (
    df_geo
    .withColumn("geolocation_lat", df_geo["geolocation_lat"].cast("double"))
    .withColumn("geolocation_lng", df_geo["geolocation_lng"].cast("double"))
)

df_geo.write.mode("overwrite").saveAsTable("olist.silver.geolocation")

In [0]:
from pyspark.sql import functions as F

df = spark.table("olist.silver.geolocation")
total = df.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.geolocation ===")
print(f"Total de linhas: {total}  (esperado: 1000163)")

print("\n1) CEPs (prefixos) diferentes")
print(f"   CEPs distintos: {df.select('geolocation_zip_code_prefix').distinct().count()}")

print("\n2) Nulos por coluna. Esperado: 0 em todas")
for c in df.columns:
    print(f"   {c}: {df.filter(df[c].isNull()).count()} nulos")

print("\n3) Linhas totalmente iguais (todas as colunas repetidas)")
print(f"   Linhas repetidas: {total - df.dropDuplicates().count()}")

print("\n4) CEPs com tamanho diferente de 5 dígitos. Esperado: 0")
print(f"   CEPs fora do padrão: {df.filter('length(geolocation_zip_code_prefix) != 5').count()}")

print("\n5) Pontos fora do Brasil (caixa aproximada, incluindo ilhas). Esperado: poucos")
fora = df.filter(
    ~df.geolocation_lat.between(-34, 6) | ~df.geolocation_lng.between(-74, -28)
).count()
print(f"   Pontos fora do Brasil: {fora}")

print("\n6) Quantos pontos existem por CEP (mínimo, média, máximo)")
display(
    df.groupBy("geolocation_zip_code_prefix").count()
      .select("count").summary("min", "mean", "max")
      .limit(3)
)

print("\n7) Estados. Esperado: 27")
print(f"   Estados distintos: {df.select('geolocation_state').distinct().count()}")

print("\n8) CEPs com mais de uma grafia de cidade")
print("   Ex.: 'sao paulo' e 'são paulo' para o mesmo CEP.")
multi = (df.groupBy("geolocation_zip_code_prefix")
           .agg(F.countDistinct("geolocation_city").alias("n"))
           .filter("n > 1").count())
print(f"   CEPs com mais de uma grafia: {multi}")



=== CHECAGEM DE QUALIDADE: olist.silver.geolocation ===
Total de linhas: 1000163  (esperado: 1000163)

1) CEPs (prefixos) diferentes
   CEPs distintos: 19015

2) Nulos por coluna. Esperado: 0 em todas
   geolocation_zip_code_prefix: 0 nulos
   geolocation_lat: 0 nulos
   geolocation_lng: 0 nulos
   geolocation_city: 0 nulos
   geolocation_state: 0 nulos

3) Linhas totalmente iguais (todas as colunas repetidas)
   Linhas repetidas: 261831

4) CEPs com tamanho diferente de 5 dígitos. Esperado: 0
   CEPs fora do padrão: 0

5) Pontos fora do Brasil (caixa aproximada, incluindo ilhas). Esperado: poucos
   Pontos fora do Brasil: 31

6) Quantos pontos existem por CEP (mínimo, média, máximo)


summary,count
min,1
mean,52.598632658427555
max,1146



7) Estados. Esperado: 27
   Estados distintos: 27

8) CEPs com mais de uma grafia de cidade
   Ex.: 'sao paulo' e 'são paulo' para o mesmo CEP.
   CEPs com mais de uma grafia: 8556


In [0]:
%sql
SHOW TABLES IN olist.silver;

database,tableName,isTemporary
silver,customers,false
silver,geolocation,false
silver,order_items,false
silver,order_payments,false
silver,order_reviews,false
silver,orders,false
silver,products,false
silver,sellers,false


In [0]:
df_customers = spark.table("olist.bronze.customers")
display(df_customers.limit(3))
df_customers.printSchema()

customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP


root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



In [0]:
df_customers = df_customers.dropDuplicates(["customer_id"])
df_customers.write.mode("overwrite").saveAsTable("olist.silver.customers")
print(df_customers.count())

99441


In [0]:
d = spark.table("olist.silver.customers")
total = d.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.customers ===")
print(f"Total de linhas: {total}  (esperado: 99441)")

print("\n1) customer_id repetido. Esperado: 0")
print(f"   Repetidos: {total - d.select('customer_id').distinct().count()}")

print("\n2) Nulos por coluna. Esperado: 0 em todas")
for c in d.columns:
    print(f"   {c}: {d.filter(d[c].isNull()).count()} nulos")

print("\n3) Clientes sem pedido em silver.orders. Esperado: 0")
orders = spark.table("olist.silver.orders")
print(f"   Clientes sem pedido: {d.join(orders, 'customer_id', 'left_anti').count()}")

print("\n4) Pessoas diferentes (customer_unique_id)")
print("   O customer_id muda a cada pedido, e o customer_unique_id identifica a pessoa.")
print(f"   Pessoas únicas: {d.select('customer_unique_id').distinct().count()}")

=== CHECAGEM DE QUALIDADE: olist.silver.customers ===
Total de linhas: 99441  (esperado: 99441)

1) customer_id repetido. Esperado: 0
   Repetidos: 0

2) Nulos por coluna. Esperado: 0 em todas
   customer_id: 0 nulos
   customer_unique_id: 0 nulos
   customer_zip_code_prefix: 0 nulos
   customer_city: 0 nulos
   customer_state: 0 nulos

3) Clientes sem pedido em silver.orders. Esperado: 0
   Clientes sem pedido: 0

4) Pessoas diferentes (customer_unique_id)
   O customer_id muda a cada pedido, e o customer_unique_id identifica a pessoa.
   Pessoas únicas: 96096


In [0]:
df_sellers = spark.table("olist.bronze.sellers")
display(df_sellers.limit(3))
df_sellers.printSchema()

seller_id,seller_zip_code_prefix,seller_city,seller_state
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: string (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)



In [0]:
df_sellers = df_sellers.dropDuplicates(["seller_id"])
df_sellers.write.mode("overwrite").saveAsTable("olist.silver.sellers")

In [0]:
from pyspark.sql import functions as F

d = spark.table("olist.silver.sellers")
total = d.count()

print("=== CHECAGEM DE QUALIDADE: olist.silver.sellers ===")
print(f"Total de linhas: {total}  (esperado: 3095)")

print("\n1) seller_id repetido. Esperado: 0")
print(f"   Repetidos: {total - d.select('seller_id').distinct().count()}")

print("\n2) Nulos por coluna. Esperado: 0 em todas")
for c in d.columns:
    print(f"   {c}: {d.filter(d[c].isNull()).count()} nulos")

print("\n3) CEPs com tamanho diferente de 5 dígitos. Esperado: 0")
print(f"   CEPs fora do padrão: {d.filter('length(seller_zip_code_prefix) != 5').count()}")

print("\n4) Estados")
print("   Esperado: menos de 27, pois nem todo estado tem vendedor.")
print(f"   Estados distintos: {d.select('seller_state').distinct().count()}")
display(d.groupBy("seller_state").count().orderBy("count", ascending=False).limit(3))

print("\n5) Vendedores de silver.order_items que não existem em silver.sellers. Esperado: 0")
itens = spark.table("olist.silver.order_items")
print(f"   Itens sem vendedor: {itens.join(d, 'seller_id', 'left_anti').count()}")

print("\n6) Vendedores que nunca aparecem em nenhum item")
print(f"   Vendedores sem venda: {d.join(itens, 'seller_id', 'left_anti').count()}")

print("\n7) Cidades com aparência suspeita (têm números, barra ou arroba)")
print("   O campo de cidade é digitado à mão e costuma ter sujeira.")
display(
    d.filter(d.seller_city.rlike("[0-9/@\\\\]"))
     .select("seller_id", "seller_city", "seller_state")
     .limit(3)
)

print("\n8) CEPs de vendedores que não existem em silver.geolocation")
geo_zips = (spark.table("olist.silver.geolocation")
              .select(F.col("geolocation_zip_code_prefix").alias("zip")).distinct())
cep_vend = d.select(F.col("seller_zip_code_prefix").alias("zip")).distinct()
print(f"   CEPs de vendedores sem geolocation: {cep_vend.join(geo_zips, 'zip', 'left_anti').count()}")

=== CHECAGEM DE QUALIDADE: olist.silver.sellers ===
Total de linhas: 3095  (esperado: 3095)

1) seller_id repetido. Esperado: 0
   Repetidos: 0

2) Nulos por coluna. Esperado: 0 em todas
   seller_id: 0 nulos
   seller_zip_code_prefix: 0 nulos
   seller_city: 0 nulos
   seller_state: 0 nulos

3) CEPs com tamanho diferente de 5 dígitos. Esperado: 0
   CEPs fora do padrão: 0

4) Estados
   Esperado: menos de 27, pois nem todo estado tem vendedor.
   Estados distintos: 23


seller_state,count
SP,1849
PR,349
MG,244



5) Vendedores de silver.order_items que não existem em silver.sellers. Esperado: 0
   Itens sem vendedor: 0

6) Vendedores que nunca aparecem em nenhum item
   Vendedores sem venda: 0

7) Cidades com aparência suspeita (têm números, barra ou arroba)
   O campo de cidade é digitado à mão e costuma ter sujeira.


seller_id,seller_city,seller_state
1cbd32d00d01bb8087a5eb088612fd9c,sp / sp,SP
6a3139c7bf09ece22a4713d956acbe5e,sao sebastiao da grama/sp,SP
7994b065a7ffb14e71c6312cf87b9de2,cariacica / es,ES



8) CEPs de vendedores que não existem em silver.geolocation
   CEPs de vendedores sem geolocation: 7
